<a href="https://colab.research.google.com/github/SaraAljuraybah/saudi-tech-job-skills-analysis/blob/main/notebooks/Modelling%20and%20Communication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3: Data Preparation for Job Role Prediction

In this phase, the goal is to prepare the cleaned dataset for a classification task that predicts job positions based on the required skills in job postings.

The main preprocessing steps include:
- selecting relevant features,
- cleaning and standardizing job titles,
- reducing the number of target classes,
- preparing the skills column in a text format suitable for feature engineering.

1.   List item
2.   List item



## Step 1: Load the cleaned dataset

The cleaned dataset from the previous phase is loaded as the starting point for modelling. This dataset already excludes duplicate records and irrelevant features, and includes normalized text fields prepared during the cleaning phase.

In [1]:
import pandas as pd

# Load the cleaned dataset from Phase 2
df = pd.read_csv("jobs_sa_cleaned.csv")

# Display dataset shape
df.shape

(1020, 22)

## Step 2: Inspect relevant columns for modelling

To build a classification model, we need:
- a target column representing the job position,
- a feature column representing the skills.

Therefore, the first step is to inspect the available columns and identify the fields that are most relevant for the modelling task.

In [2]:
# Display all column names
df.columns.tolist()

['job_id',
 'job_title',
 'employer_name',
 'job_publisher',
 'job_employment_type',
 'job_employment_types',
 'job_apply_link',
 'job_apply_is_direct',
 'apply_options',
 'job_description',
 'job_is_remote',
 'job_posted_at',
 'job_location',
 'job_city',
 'job_country',
 'job_latitude',
 'job_longitude',
 'job_google_link',
 'job_onet_soc',
 'job_onet_job_zone',
 'search_query',
 'clean_description']

In [3]:
# Preview key columns
df[["job_title", "job_description"]].head()

,job_title,job_description
0,Senior Specialist – Data Science,Be the change. Join the world’s most visionary...
1,Data Scientist I,About Mozn\n\nMOZN is a leading Enterprise AI ...
2,Data Science Intern,About MOZN\n\nMOZN is a leading Enterprise AI ...
3,Strategic Data Scientist — Predictive Analytic...,A data solutions company located in Riyadh is ...
4,Data Scientist,The Data Scientist is responsible for deliveri...


## Step 3: Select the columns needed for modelling

For this task, the model input should represent the required skills, while the output should represent the job position.

The selected columns are:
- `job_title` as the target label,
- a text-based skills field as the input feature.

In [4]:
model_df = df[["job_title", "clean_description"]].copy()

model_df.head()

,job_title,clean_description
0,Senior Specialist – Data Science,be the change join the worlds most visionary d...
1,Data Scientist I,about mozn mozn is a leading enterprise ai com...
2,Data Science Intern,about mozn mozn is a leading enterprise ai com...
3,Strategic Data Scientist — Predictive Analytic...,a data solutions company located in riyadh is ...
4,Data Scientist,the data scientist is responsible for deliveri...


In [5]:
model_df["job_title"].value_counts().head(20)

,count
job_title,
AI Engineer,6
Backend Developer,6
"Remote Senior Data Engineer - ETL, Python & ML Architect",6
Senior Data Engineer,6
Data Scientist,5
"Software Engineer - Backend (Mid-Level, Remote)",5
"Senior Data Engineer - Remote, Flexible Hours & Growth",5
Senior Software Engineer,4
Data Analyst,4


## Step 4: Clean and standardize job titles

Job titles contain a large number of variations, including seniority levels, locations, and additional descriptions.

To reduce noise and improve model performance, job titles are cleaned by removing unnecessary words and standardizing the text format.

In [6]:
import re

def clean_job_title(title):
    title = str(title).lower()

    # Remove common extra words
    remove_words = [
        "senior", "junior", "lead", "principal", "intern",
        "remote", "mid-level", "mid level", "entry level",
        "manager", "specialist", "sr", "jr"
    ]

    for word in remove_words:
        title = title.replace(word, "")

    # Remove symbols and extra spaces
    title = re.sub(r'[^a-zA-Z\s]', ' ', title)
    title = re.sub(r'\s+', ' ', title).strip()

    return title

model_df["clean_title"] = model_df["job_title"].apply(clean_job_title)

model_df[["job_title", "clean_title"]].head(10)

,job_title,clean_title
0,Senior Specialist – Data Science,data science
1,Data Scientist I,data scientist i
2,Data Science Intern,data science
3,Strategic Data Scientist — Predictive Analytic...,strategic data scientist predictive analytics ...
4,Data Scientist,data scientist
5,Data Scientist - Digital & AI Projects (Saudi ...,data scientist digital ai projects saudi arabia
6,Data Scientist Specialist,data scientist
7,Data Analyst - Junior level,data analyst level
8,Lead Data Scientist (Fintech / Banking),data scientist fintech banking
9,Expert Data Scientist,expert data scientist


## Step 5: Map job titles into unified role categories

After cleaning, similar job titles are grouped into a smaller number of consistent role categories.

This helps reduce class imbalance and improves the performance of the classification model.

In [7]:
def map_role(title):
    title = title.lower()

    if "data scientist" in title:
        return "data scientist"

    elif "data analyst" in title:
        return "data analyst"

    elif "data engineer" in title:
        return "data engineer"

    elif "ai" in title or "machine learning" in title or "ml" in title:
        return "ai/ml engineer"

    else:
        return "other"

model_df["role_label"] = model_df["clean_title"].apply(map_role)

model_df["role_label"].value_counts()

,count
role_label,
other,580
ai/ml engineer,194
data engineer,96
data analyst,84
data scientist,66


## Step 6: Filter relevant job roles

The dataset initially contained a large number of job roles grouped under the "other" category, which includes unrelated positions such as software and frontend roles.

Since the focus of this study is on data and AI-related jobs, only the following roles were retained:
- Data Scientist
- Data Analyst
- Data Engineer
- AI/ML Engineer

All other roles were removed to ensure a more focused and balanced classification task.

In [8]:
top_roles = ["data scientist", "data analyst", "data engineer", "ai/ml engineer"]

model_df = model_df[model_df["role_label"].isin(top_roles)].copy()

model_df["role_label"].value_counts()

,count
role_label,
ai/ml engineer,194
data engineer,96
data analyst,84
data scientist,66


In [11]:
model_df.shape

(440, 4)

## Step 7: Prepare text data for modelling

The cleaned job description is used as the input feature for the classification model.

To ensure data quality, missing and empty text values are removed. The column is then renamed to "skills_text" to clearly represent its role as the input feature.

In [12]:
# Rename column for clarity
model_df = model_df.rename(columns={"clean_description": "skills_text"})

# Remove missing values
model_df = model_df.dropna(subset=["skills_text", "role_label"])

# Remove empty text
model_df = model_df[model_df["skills_text"].str.strip() != ""]

# Check final shape
model_df.shape

(440, 4)

## Step 8: Export modelling dataset

The final dataset is exported and will be used in the next phase for feature engineering and model training.

In [13]:
model_df.to_csv("jobs_sa_model_ready.csv", index=False)